# Notebook 2 — Meridian MMM Model

Fits a Bayesian Marketing Mix Model using Google's Meridian library on the synthetic data from Notebook 1.

**Run Notebook 1 first** to generate `data/synthetic_mmm.csv`.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

import meridian
from meridian.data import input_data as input_data_lib
from meridian.model import meridian as meridian_model
from meridian.model import spec
from meridian.analysis import optimizer
from meridian.analysis import analyzer

sns.set_theme(style='whitegrid')
%matplotlib inline

print('Meridian version:', meridian.__version__)

## 1. Load data & build xarray InputData

In [ ]:
df = pd.read_csv('../data/synthetic_mmm.csv', parse_dates=['date'])

GEO = 'national'
MEDIA_CHANNELS = ['tv', 'search', 'social', 'display']
SPEND_COLS     = ['tv_spend', 'search_spend', 'social_spend', 'display_spend']
N_WEEKS = len(df)

# Meridian expects dims: (geo, time) for kpi & (geo, time, media_channel) for media
kpi = xr.DataArray(
    data=df['revenue'].values[np.newaxis, :],          # shape (1, 104)
    dims=['geo', 'time'],
    coords={'geo': [GEO], 'time': df['date'].values}
)

# Stack spend channels into (geo, time, media_channel)
media_spend = xr.DataArray(
    data=df[SPEND_COLS].values[np.newaxis, :, :],      # shape (1, 104, 4)
    dims=['geo', 'time', 'media_channel'],
    coords={'geo': [GEO], 'time': df['date'].values, 'media_channel': MEDIA_CHANNELS}
)

# Use spend as the media impressions proxy (scale to 0-1 per channel)
media_impressions = media_spend / media_spend.max(dim='time')

input_data = input_data_lib.InputData(
    kpi=kpi,
    kpi_type='revenue',
    media=media_impressions,
    media_spend=media_spend,
    revenue_per_kpi=1.0,   # revenue IS the KPI, so multiplier = 1
)

print('KPI shape:         ', kpi.shape)
print('Media shape:       ', media_impressions.shape)
print('Media spend shape: ', media_spend.shape)

## 2. Define ModelSpec & fit

In [ ]:
model_spec = spec.ModelSpec(
    # Hill saturation priors (ec = half-saturation, slope = shape)
    hill_before_adstock=False,   # adstock first, then saturation
    # Adstock / lag priors use defaults
)

mmm = meridian_model.Meridian(input_data=input_data, model_spec=model_spec)

In [ ]:
# MCMC sampling — use fewer draws for a quick run; increase for production
mmm.sample_posterior(
    n_chains=2,
    n_adapt=500,
    n_burnin=500,
    n_keep=1000,
    seed=42,
)
print('Sampling complete.')

## 3. Model diagnostics

In [ ]:
# R-hat convergence check (want < 1.05 for all params)
rhat = mmm.get_rhat()
print('Max R-hat:', float(rhat.max()))
rhat

In [ ]:
# Posterior predictive: actual vs fitted revenue
an = analyzer.Analyzer(mmm)

predicted = an.expected_outcome(use_posterior=True)  # shape (chains*draws, geo, time)
pred_mean = predicted.mean(dim=['chain', 'draw']).sel(geo=GEO).values
actual    = df['revenue'].values

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df['date'], actual,    label='Actual',    color='steelblue', linewidth=1.5)
ax.plot(df['date'], pred_mean, label='Fitted',    color='tomato',    linewidth=1.5, linestyle='--')
ax.set_title('Actual vs Fitted Revenue', fontsize=13)
ax.legend()
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x:,.0f}k'))
plt.tight_layout()
plt.savefig('../outputs/actual_vs_fitted.png', dpi=150)
plt.show()

r2 = np.corrcoef(actual, pred_mean)[0,1]**2
print(f'R²: {r2:.3f}')

## 4. Channel ROI & contribution

In [ ]:
roi = an.roi(use_posterior=True)  # DataArray: (chain, draw, media_channel)

roi_df = pd.DataFrame({
    'channel': MEDIA_CHANNELS,
    'roi_mean': float(roi.mean(dim=['chain','draw']).values) if roi.ndim == 0
               else roi.mean(dim=['chain','draw']).values,
    'roi_lo':   roi.quantile(0.1,  dim=['chain','draw']).values,
    'roi_hi':   roi.quantile(0.9,  dim=['chain','draw']).values,
})

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
bars = ax.barh(roi_df['channel'], roi_df['roi_mean'], color=colors, alpha=0.8)
ax.errorbar(
    roi_df['roi_mean'], roi_df['channel'],
    xerr=[roi_df['roi_mean'] - roi_df['roi_lo'], roi_df['roi_hi'] - roi_df['roi_mean']],
    fmt='none', color='black', capsize=4
)
ax.axvline(1.0, color='grey', linestyle='--', linewidth=1, label='Break-even (ROI=1)')
ax.set_title('Channel ROI (80% credible interval)', fontsize=13)
ax.set_xlabel('£ revenue per £1 spend')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/channel_roi.png', dpi=150)
plt.show()

roi_df.round(2)

## 5. Revenue decomposition (waterfall)

In [ ]:
contrib = an.media_contribution_hat(use_posterior=True)  # (chain, draw, geo, time, channel)
contrib_mean = contrib.mean(dim=['chain','draw']).sel(geo=GEO).sum(dim='time')  # per channel total

baseline_total = float(an.baseline_hat(use_posterior=True).mean(dim=['chain','draw']).sel(geo=GEO).sum())
channel_totals = contrib_mean.values

labels  = ['Baseline'] + MEDIA_CHANNELS
values  = [baseline_total] + list(channel_totals)
shares  = np.array(values) / sum(values) * 100

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#95a5a6'] + colors
bars = ax.bar(labels, shares, color=bar_colors, alpha=0.85, edgecolor='white')
for bar, s in zip(bars, shares):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{s:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_title('Revenue Decomposition by Driver', fontsize=13)
ax.set_ylabel('% of Total Revenue')
ax.set_ylim(0, max(shares) * 1.15)
plt.tight_layout()
plt.savefig('../outputs/revenue_decomposition.png', dpi=150)
plt.show()

## 6. Response curves

In [ ]:
spend_grid = np.linspace(0, 300, 200)  # £000s

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, channel, color in zip(axes.flat, MEDIA_CHANNELS, colors):
    curves = an.response_curves(
        media_channel=channel,
        spend_grid=spend_grid,
        use_posterior=True,
    )  # shape (chain*draw, spend_grid)
    lo  = np.quantile(curves, 0.1, axis=0)
    mid = np.quantile(curves, 0.5, axis=0)
    hi  = np.quantile(curves, 0.9, axis=0)

    ax.fill_between(spend_grid, lo, hi, alpha=0.2, color=color)
    ax.plot(spend_grid, mid, color=color, linewidth=2)
    ax.set_title(channel.title(), fontsize=12)
    ax.set_xlabel('Weekly spend (£000s)')
    ax.set_ylabel('Incremental revenue (£000s)')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x:,.0f}k'))

fig.suptitle('Diminishing Returns Response Curves (80% CI)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/response_curves.png', dpi=150)
plt.show()

## 7. Budget optimisation

Given the same total budget, find the spend allocation that maximises revenue.

In [ ]:
total_budget = float(df[SPEND_COLS].sum().sum())   # same total as historical

opt = optimizer.BudgetOptimizer(mmm)
result = opt.optimize(
    budget=total_budget,
    use_posterior=True,
)

opt_spend    = result.optimal_spend    # per channel
current_spend = df[SPEND_COLS].sum().values

opt_df = pd.DataFrame({
    'channel':  MEDIA_CHANNELS,
    'current':  current_spend,
    'optimal':  opt_spend,
    'change':   opt_spend - current_spend,
    'change_pct': (opt_spend - current_spend) / current_spend * 100,
})

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(MEDIA_CHANNELS))
w = 0.35
ax.bar(x - w/2, opt_df['current'], w, label='Current',  color='#95a5a6', alpha=0.85)
ax.bar(x + w/2, opt_df['optimal'], w, label='Optimal',  color='#3498db', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(MEDIA_CHANNELS)
ax.set_title('Current vs Optimal Budget Allocation', fontsize=13)
ax.set_ylabel('Total spend (£000s)')
ax.legend()
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x:,.0f}k'))
plt.tight_layout()
plt.savefig('../outputs/budget_optimisation.png', dpi=150)
plt.show()

opt_df.round(1)